# MOML Baseline Demo
Tests all Day 1 components: data loading, model architectures, and training/evaluation.

In [ ]:
import torch
import time
import numpy as np
from data_loader import get_dataloaders, get_dataset_info, DEVICE, DEFAULT_TRAIN_SUBSET
from models import build_model, count_parameters, _ARCH_REGISTRY
from train_eval import train_and_evaluate

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Device          : {DEVICE}")
print(f"Train subset    : {DEFAULT_TRAIN_SUBSET:,} samples")

## 2. Data Loading
Both datasets loaded locally. CUDA-aware: 20K on GPU, 10K on CPU.

In [ ]:
for ds_name in ["cifar10", "fashion_mnist"]:
    info = get_dataset_info(ds_name)
    # Use num_workers=0 to avoid multiprocessing issues on Windows in notebooks
    train_loader, test_loader = get_dataloaders(ds_name, batch_size=64, test_subset_size=500, seed=42, num_workers=0)
    imgs, lbls = next(iter(train_loader))
    
    print(f"\n--- {ds_name} ---")
    print(f"  Classes      : {info['num_classes']}  {info['class_names']}")
    print(f"  Channels     : {info['input_channels']}")
    print(f"  Resolution   : {info['default_resolution']}x{info['default_resolution']}")
    print(f"  Train samples: {len(train_loader.dataset):,}")
    print(f"  Test samples : {len(test_loader.dataset):,}")
    print(f"  Batch shape  : {imgs.shape}")
    print(f"  Pixel range  : [{imgs.min():.3f}, {imgs.max():.3f}]")

## 3. Architecture Exploration
Compare all 3 CNN families across small → large configs.

In [ ]:
configs = [
    ("plain",                2, 16,  64,  0.1),   # small
    ("plain",                4, 64,  256, 0.3),   # large
    ("residual",             2, 16,  64,  0.1),
    ("residual",             4, 64,  256, 0.3),
    ("depthwise_separable",  2, 16,  64,  0.1),
    ("depthwise_separable",  4, 64,  256, 0.3),
]

print(f"{'Arch':>22s} | {'Layers':>6s} | {'Ch':>4s} | {'FC':>4s} | {'Params':>10s} | {'Output':>12s}")
print("-" * 75)

for arch, layers, ch, fc, drop in configs:
    model = build_model(
        arch_type=arch, input_channels=3, input_resolution=32,
        num_classes=10, num_conv_layers=layers, num_channels=ch,
        num_fc_units=fc, dropout_rate=drop,
    ).to(DEVICE)
    
    x = torch.randn(4, 3, 32, 32, device=DEVICE)
    out = model(x)
    params = count_parameters(model)
    
    print(f"{arch:>22s} | {layers:>6d} | {ch:>4d} | {fc:>4d} | {params:>10,} | {str(tuple(out.shape)):>12s}")

## 4. Parameter Count Comparison
Shows why DepthSep is great for the compactness objective.

In [ ]:
print(f"\nSame config (4 layers, 64 channels, 256 FC) across architectures:\n")
for arch in ["plain", "residual", "depthwise_separable"]:
    m = build_model(arch_type=arch, input_channels=3, input_resolution=32,
                    num_classes=10, num_conv_layers=4, num_channels=64,
                    num_fc_units=256, dropout_rate=0.2)
    p = count_parameters(m)
    print(f"  {arch:>22s}: {p:>10,} params")

## 5. Full Train + Evaluate (3 objectives)
Trains one model per architecture on each dataset. Uses a small subset (5K) for speed.

In [ ]:
trial_configs = [
    {"arch_type": "plain",                "num_conv_layers": 2, "num_channels": 32,
     "num_fc_units": 128, "learning_rate": 1e-3, "batch_size": 64,
     "num_epochs": 5, "dropout_rate": 0.2, "optimizer_type": "Adam", "input_resolution": 32},
    
    {"arch_type": "residual",             "num_conv_layers": 3, "num_channels": 32,
     "num_fc_units": 128, "learning_rate": 1e-3, "batch_size": 64,
     "num_epochs": 5, "dropout_rate": 0.2, "optimizer_type": "Adam", "input_resolution": 32},
    
    {"arch_type": "depthwise_separable",  "num_conv_layers": 3, "num_channels": 32,
     "num_fc_units": 128, "learning_rate": 1e-3, "batch_size": 64,
     "num_epochs": 5, "dropout_rate": 0.2, "optimizer_type": "Adam", "input_resolution": 32},
]

all_results = []

for dataset in ["cifar10", "fashion_mnist"]:
    print(f"\n{'='*70}")
    print(f"Dataset: {dataset}")
    print(f"{'='*70}")
    print(f"{'Arch':>22s} | {'Accuracy':>8s} | {'Infer(ms)':>9s} | {'Params':>10s} | {'Time':>6s}")
    print("-" * 70)
    
    for cfg in trial_configs:
        t0 = time.time()
        result = train_and_evaluate(config=cfg, dataset_name=dataset, seed=42, train_subset_size=5_000)
        elapsed = time.time() - t0
        
        all_results.append({"dataset": dataset, **cfg, **result})
        
        print(
            f"{cfg['arch_type']:>22s} | "
            f"{result['accuracy']:>7.2%} | "
            f"{result['inference_ms']:>8.3f} | "
            f"{result['param_count']:>10,} | "
            f"{elapsed:>5.1f}s"
        )

## 6. Trade-off Visualization (Quick Scatter)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for dataset in ["cifar10", "fashion_mnist"]:
    subset = [r for r in all_results if r["dataset"] == dataset]
    
    accs    = [r["accuracy"] * 100 for r in subset]
    infers  = [r["inference_ms"] for r in subset]
    params  = [r["param_count"] / 1000 for r in subset]  # in K
    labels  = [r["arch_type"] for r in subset]
    colors  = {"plain": "#e74c3c", "residual": "#3498db", "depthwise_separable": "#2ecc71"}
    marker  = "o" if dataset == "cifar10" else "s"
    
    for i, (a, inf, p, lbl) in enumerate(zip(accs, infers, params, labels)):
        axes[0].scatter(p, a, c=colors[lbl], marker=marker, s=80, edgecolors="k", linewidths=0.5)
        axes[1].scatter(inf, a, c=colors[lbl], marker=marker, s=80, edgecolors="k", linewidths=0.5)
        axes[2].scatter(p, inf, c=colors[lbl], marker=marker, s=80, edgecolors="k", linewidths=0.5)

axes[0].set_xlabel("Params (K)"); axes[0].set_ylabel("Accuracy (%)"); axes[0].set_title("Accuracy vs Size")
axes[1].set_xlabel("Inference (ms)"); axes[1].set_ylabel("Accuracy (%)"); axes[1].set_title("Accuracy vs Speed")
axes[2].set_xlabel("Params (K)"); axes[2].set_ylabel("Inference (ms)"); axes[2].set_title("Speed vs Size")

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=8, label='Plain'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#3498db', markersize=8, label='Residual'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=8, label='DepthSep'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='gray', markersize=8, label='CIFAR-10'),
    Line2D([0],[0], marker='s', color='w', markerfacecolor='gray', markersize=8, label='FashionMNIST'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.08))
plt.tight_layout()
plt.show()